# EvidenceLab — Notebook 03 FINAL
## LoRA + QLoRA · Llama 3.2 3B Instruct · Colab T4

**Por ahora ejecuta únicamente la primera celda.**

Cuando confirme acceso a Llama seguimos con el resto.

Preparado para:
- dataset v3 `sft_v3_contract_augmented`;
- 412 train / 76 validation / 137 test;
- `MAX_LENGTH=1280`;
- LoRA normal;
- QLoRA NF4 4-bit;
- ranks `4, 8, 16, 32, 64`;
- target modules `q/k/v/o/gate/up/down_proj`;
- mismas 5 pruebas de test;
- métricas de JSON/schema;
- merge del adapter ganador;
- tamaños en disco y comparación posterior contra Phi.

In [1]:

!pip -q install -U "transformers<5" "trl>=0.23,<1" "peft>=0.17,<1" datasets accelerate bitsandbytes mlflow psutil
!pip -q uninstall -y torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265

In [5]:
import gc
import json
import os
import random
import time
from collections import Counter
from pathlib import Path

import mlflow
import pandas as pd
import psutil
import torch

from datasets import Dataset
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), (
    "Activa una GPU T4 en Colab."
)

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)

print("GPU:", GPU_NAME)
print(f"VRAM: {VRAM_GB:.2f} GB")
print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)

GPU: Tesla T4
VRAM: 14.56 GB
CUDA: 12.8
PyTorch: 2.11.0+cu128


In [6]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
print("HF listo")

HF listo


In [7]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/EvidenceLab/EvidenceLab"
)

DERIVED_DIR = (
    DRIVE_ROOT
    / "derived"
    / "sft_v3_contract_augmented"
)

ARTIFACTS_DIR = (
    DRIVE_ROOT
    / "artifacts"
    / "03_llama32_lora_qlora"
)

if not DERIVED_DIR.exists():
    raise FileNotFoundError(
        f"No encontré: {DERIVED_DIR}"
    )

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Dataset:", DERIVED_DIR)
print("Artefactos:", ARTIFACTS_DIR)

Mounted at /content/drive
Dataset: /content/drive/MyDrive/EvidenceLab/EvidenceLab/derived/sft_v3_contract_augmented
Artefactos: /content/drive/MyDrive/EvidenceLab/EvidenceLab/artifacts/03_llama32_lora_qlora


# 2. Configuración común

In [8]:
SEED = 42
set_seed(SEED)
random.seed(SEED)

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
MAX_LENGTH = 1280

TRAIN_BATCH = 1
EVAL_BATCH = 1
GRAD_ACCUM = 4

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

QLORA_RANKS = [4, 8, 16, 32, 64]

LORA_LR = 2e-4
QLORA_LR = 2e-4
FINAL_EPOCHS = 1
RANK_SWEEP_MAX_STEPS = 60

MLRUNS_DIR = DRIVE_ROOT / "mlruns"
MLRUNS_DIR.mkdir(parents=True, exist_ok=True)

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri(
    MLRUNS_DIR.resolve().as_uri()
)
mlflow.set_experiment(
    "EvidenceLab_Llama32_LoRA_QLoRA"
)

<Experiment: artifact_location='file:///content/drive/MyDrive/EvidenceLab/EvidenceLab/mlruns/141740923460021963', creation_time=1787242159424, effective_trace_archival_retention=None, experiment_id='141740923460021963', last_update_time=1787242159424, lifecycle_stage='active', name='EvidenceLab_Llama32_LoRA_QLoRA', tags={}, trace_location=None, workspace='default'>

# 3. Dataset v3

In [9]:
def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

train_rows = load_jsonl(
    DERIVED_DIR / "train_contract.jsonl"
)
val_rows = load_jsonl(
    DERIVED_DIR / "validation_contract.jsonl"
)
test_rows = load_jsonl(
    DERIVED_DIR
    / "test_contract_DO_NOT_TRAIN.jsonl"
)

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)
test_ds = Dataset.from_list(test_rows)

assert len(train_ds) == 412
assert len(val_ds) == 76
assert len(test_ds) == 137

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))

Train: 412
Validation: 76
Test: 137


In [10]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

# 4. Helpers y cinco prompts fijos

In [11]:
FIXED_TEST_IDS = [
    "CASE-MX-006-VERIFY-001",
    "CASE-MX-006-THEORY-001",
    "CASE-MX-006-CONTRA-001",
    "CASE-MX-006-RESP-001",
    "CASE-MX-006-RECONSTRUCTION",
]

test_by_id = {
    row["example_id"]: row
    for row in test_rows
}

fixed_test = [
    test_by_id[x]
    for x in FIXED_TEST_IDS
]

def expected_text(example):
    return example["completion"][0]["content"]

def generate_from_prompt(
    model,
    prompt_messages,
    max_new_tokens=650,
):
    model.eval()

    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output[
        0,
        inputs["input_ids"].shape[1]:,
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

# 5. Configuración SFT común

In [12]:
def make_sft_config(
    output_dir,
    learning_rate,
    *,
    max_steps=-1,
    epochs=1,
    run_name="llama32-sft",
):
    return SFTConfig(
        output_dir=str(output_dir),
        max_length=MAX_LENGTH,
        packing=True,
        packing_strategy="wrapped",
        padding_free=False,
        eval_packing=False,
        completion_only_loss=True,
        num_train_epochs=epochs,
        max_steps=max_steps,
        per_device_train_batch_size=TRAIN_BATCH,
        per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=learning_rate,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        },
        fp16=True,
        bf16=False,
        eval_strategy="no",
        logging_strategy="steps",
        logging_steps=1,
        save_strategy="no",
        report_to=["mlflow"],
        seed=SEED,
        data_seed=SEED,
        run_name=run_name,
    )

# 6. LoRA normal

In [13]:
def load_lora_base():
    gc.collect()
    torch.cuda.empty_cache()

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        dtype=torch.float16,
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
    )

    model.config.use_cache = False
    return model

LORA_CONFIG = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

## 6.1 Smoke LoRA

In [14]:
lora_smoke_model = load_lora_base()

lora_smoke_trainer = SFTTrainer(
    model=lora_smoke_model,
    args=make_sft_config(
        "/content/lora_smoke",
        learning_rate=LORA_LR,
        max_steps=1,
        epochs=1,
        run_name="lora-smoke",
    ),
    train_dataset=train_ds.select(range(8)),
    processing_class=tokenizer,
    peft_config=LORA_CONFIG,
)

torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()

result = lora_smoke_trainer.train()

elapsed = time.perf_counter() - start
peak = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("LORA SMOKE OK")
print("Loss:", result.training_loss)
print(f"Tiempo: {elapsed:.2f} s")
print(f"VRAM pico: {peak:.2f} GB")

del lora_smoke_trainer, lora_smoke_model
gc.collect()
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
1,1.086000


LORA SMOKE OK
Loss: 1.0859615802764893
Tiempo: 12.29 s
VRAM pico: 8.25 GB


## 6.2 LoRA final r=16

In [15]:
LORA_DIR = ARTIFACTS_DIR / "lora_r16"

lora_model = load_lora_base()

lora_trainer = SFTTrainer(
    model=lora_model,
    args=make_sft_config(
        "/content/lora_final",
        learning_rate=LORA_LR,
        epochs=FINAL_EPOCHS,
        run_name="lora-r16-final",
    ),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=LORA_CONFIG,
)

torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()

lora_train = lora_trainer.train()
lora_eval = lora_trainer.evaluate()

lora_time = time.perf_counter() - start
lora_peak = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)
lora_eval_loss = float(
    lora_eval["eval_loss"]
)

print("LoRA eval loss:", lora_eval_loss)
print(f"Tiempo: {lora_time/60:.2f} min")
print(f"VRAM pico: {lora_peak:.2f} GB")

lora_trainer.model.save_pretrained(
    LORA_DIR
)
tokenizer.save_pretrained(LORA_DIR)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/412 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/412 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/76 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/76 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
1,0.665900
2,0.605500
3,0.908400
4,0.624200
5,0.701800
6,0.943200
7,0.501000
8,0.543500
9,0.755000
10,0.493600


LoRA eval loss: 0.1567792445421219
Tiempo: 11.13 min
VRAM pico: 8.43 GB


('/content/drive/MyDrive/EvidenceLab/EvidenceLab/artifacts/03_llama32_lora_qlora/lora_r16/tokenizer_config.json',
 '/content/drive/MyDrive/EvidenceLab/EvidenceLab/artifacts/03_llama32_lora_qlora/lora_r16/special_tokens_map.json',
 '/content/drive/MyDrive/EvidenceLab/EvidenceLab/artifacts/03_llama32_lora_qlora/lora_r16/chat_template.jinja',
 '/content/drive/MyDrive/EvidenceLab/EvidenceLab/artifacts/03_llama32_lora_qlora/lora_r16/tokenizer.json')

## 6.3 Evaluación LoRA

In [16]:
lora_outputs = []

for row in fixed_test:
    pred = generate_from_prompt(
        lora_trainer.model,
        row["prompt"],
    )

    lora_outputs.append({
        "example_id": row["example_id"],
        "task": row["task"],
        "lora_after": pred,
        "expected": expected_text(row),
    })

lora_df = pd.DataFrame(lora_outputs)

lora_df.to_csv(
    ARTIFACTS_DIR
    / "lora_five_prompts.csv",
    index=False,
)

display(lora_df)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,example_id,task,lora_after,expected
0,CASE-MX-006-VERIFY-001,claim_verification,"{""verdict"": ""supported"", ""reason"": ""La clasifi...","{""verdict"": ""supported"", ""reason"": ""La clasifi..."
1,CASE-MX-006-THEORY-001,theory_assessment,"{""assessment"": ""contradicted"", ""supporting_ele...","{""assessment"": ""contradicted"", ""supporting_ele..."
2,CASE-MX-006-CONTRA-001,contradiction_analysis,"{""relation"": ""tension_or_contradiction"", ""expl...","{""relation"": ""tension_or_contradiction"", ""expl..."
3,CASE-MX-006-RESP-001,responsibility_reasoning,"{""official_outcome"": ""revocacion_por_presuncio...","{""official_outcome"": ""revocacion_por_presuncio..."
4,CASE-MX-006-RECONSTRUCTION,case_reconstruction,"{""case_title"": ""Homicidio con testigo retracta...","{""case_title"": ""Homicidio con testigo retracta..."


In [17]:
del lora_trainer, lora_model
gc.collect()
torch.cuda.empty_cache()

# 7. QLoRA NF4 4-bit

In [18]:
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

def load_qlora_base():
    gc.collect()
    torch.cuda.empty_cache()

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        quantization_config=BNB_CONFIG,
        dtype=torch.float16,          # <-- esta línea faltaba
        device_map={"": 0},
        attn_implementation="sdpa",
    )

    model.config.use_cache = False

    return prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
    )

def qlora_config(rank):
    return LoraConfig(
        r=rank,
        lora_alpha=max(2 * rank, 8),
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=TARGET_MODULES,
    )

def cast_trainable_to_fp32(model):
    changed = 0
    for _, p in model.named_parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.to(torch.float32)
            changed += 1
    print("adapters pasados a fp32:", changed)
    return model

## 7.1 Sweep ranks 4, 8, 16, 32, 64

In [20]:
rank_results = []

for rank in QLORA_RANKS:
    print("\n" + "=" * 60)
    print("QLoRA rank =", rank)

    model = load_qlora_base()

    trainer_rank = SFTTrainer(
        model=model,
        args=make_sft_config(
            f"/content/qlora_rank_{rank}",
            learning_rate=QLORA_LR,
            max_steps=RANK_SWEEP_MAX_STEPS,
            epochs=1,
            run_name=f"qlora-r{rank}",
        ),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        peft_config=qlora_config(rank),
    )


    cast_trainable_to_fp32(trainer_rank.model)   # <-- aquí
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()

    tr = trainer_rank.train()
    ev = trainer_rank.evaluate()

    elapsed = time.perf_counter() - start
    peak = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    rank_results.append({
        "rank": rank,
        "train_loss": float(
            tr.training_loss
        ),
        "eval_loss": float(
            ev["eval_loss"]
        ),
        "time_seconds": elapsed,
        "peak_vram_gb": peak,
    })

    pd.DataFrame(
        rank_results
    ).to_csv(
        ARTIFACTS_DIR
        / "qlora_rank_sweep_partial.csv",
        index=False,
    )

    del trainer_rank, model
    gc.collect()
    torch.cuda.empty_cache()

rank_df = pd.DataFrame(rank_results)

rank_df = (
    rank_df
    .sort_values("eval_loss")
    .reset_index(drop=True)
)

display(rank_df)

BEST_QLORA_RANK = int(
    rank_df.iloc[0]["rank"]
)

print("BEST_QLORA_RANK:", BEST_QLORA_RANK)

rank_df.to_csv(
    ARTIFACTS_DIR
    / "qlora_rank_sweep.csv",
    index=False,
)


QLoRA rank = 4


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/412 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/412 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/76 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


adapters pasados a fp32: 392


Step,Training Loss
1,0.667300
2,0.541100
3,0.838600
4,0.502900
5,0.480000
6,0.845100
7,0.307900
8,0.400500
9,0.580600
10,0.315200


,rank,train_loss,eval_loss,time_seconds,peak_vram_gb
0,64,0.246683,0.092420,880.595448,7.302553
1,32,0.288787,0.103463,875.229999,6.559389
2,16,0.345758,0.131411,873.041418,6.197084
3,8,0.410178,0.183777,872.076080,6.015932
4,4,0.491836,0.286350,869.597964,5.924867


BEST_QLORA_RANK: 64


## 7.2 QLoRA final

In [21]:
QLORA_DIR = (
    ARTIFACTS_DIR
    / f"qlora_r{BEST_QLORA_RANK}"
)

qlora_model = load_qlora_base()

qlora_trainer = SFTTrainer(
    model=qlora_model,
    args=make_sft_config(
        "/content/qlora_final",
        learning_rate=QLORA_LR,
        epochs=FINAL_EPOCHS,
        run_name=(
            f"qlora-final-r"
            f"{BEST_QLORA_RANK}"
        ),
    ),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=qlora_config(
        BEST_QLORA_RANK
    ),
)
cast_trainable_to_fp32(qlora_trainer.model)   # <-- aquí
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()

qlora_train = qlora_trainer.train()
qlora_eval = qlora_trainer.evaluate()

qlora_time = time.perf_counter() - start
qlora_peak = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)
qlora_eval_loss = float(
    qlora_eval["eval_loss"]
)

print("QLoRA eval loss:", qlora_eval_loss)
print(f"Tiempo: {qlora_time/60:.2f} min")
print(f"VRAM pico: {qlora_peak:.2f} GB")

qlora_trainer.model.save_pretrained(
    QLORA_DIR
)
tokenizer.save_pretrained(QLORA_DIR)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/412 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/412 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/76 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/76 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


adapters pasados a fp32: 392


Step,Training Loss
1,0.667300
2,0.541100
3,0.838700
4,0.503200
5,0.480800
6,0.845600
7,0.309000
8,0.400400
9,0.581500
10,0.316300


QLoRA eval loss: 0.09752330929040909
Tiempo: 13.07 min
VRAM pico: 7.30 GB


('/content/drive/MyDrive/EvidenceLab/EvidenceLab/artifacts/03_llama32_lora_qlora/qlora_r64/tokenizer_config.json',
 '/content/drive/MyDrive/EvidenceLab/EvidenceLab/artifacts/03_llama32_lora_qlora/qlora_r64/special_tokens_map.json',
 '/content/drive/MyDrive/EvidenceLab/EvidenceLab/artifacts/03_llama32_lora_qlora/qlora_r64/chat_template.jinja',
 '/content/drive/MyDrive/EvidenceLab/EvidenceLab/artifacts/03_llama32_lora_qlora/qlora_r64/tokenizer.json')

## 7.3 Evaluación QLoRA

In [22]:
qlora_outputs = []

for row in fixed_test:
    pred = generate_from_prompt(
        qlora_trainer.model,
        row["prompt"],
    )

    qlora_outputs.append({
        "example_id": row["example_id"],
        "task": row["task"],
        "qlora_after": pred,
        "expected": expected_text(row),
    })

qlora_df = pd.DataFrame(
    qlora_outputs
)

qlora_df.to_csv(
    ARTIFACTS_DIR
    / "qlora_five_prompts.csv",
    index=False,
)

display(qlora_df)

,example_id,task,qlora_after,expected
0,CASE-MX-006-VERIFY-001,claim_verification,"{""verdict"": ""supported"", ""reason"": ""La clasifi...","{""verdict"": ""supported"", ""reason"": ""La clasifi..."
1,CASE-MX-006-THEORY-001,theory_assessment,"{""assessment"": ""contradicted"", ""supporting_ele...","{""assessment"": ""contradicted"", ""supporting_ele..."
2,CASE-MX-006-CONTRA-001,contradiction_analysis,"{""relation"": ""tension_or_contradiction"", ""expl...","{""relation"": ""tension_or_contradiction"", ""expl..."
3,CASE-MX-006-RESP-001,responsibility_reasoning,"{""official_outcome"": ""revocacion_por_presuncio...","{""official_outcome"": ""revocacion_por_presuncio..."
4,CASE-MX-006-RECONSTRUCTION,case_reconstruction,"{""case_title"": ""Homicidio con testigo retracta...","{""case_title"": ""Homicidio con testigo retracta..."


# 8. Merge del QLoRA ganador

In [23]:
del qlora_trainer, qlora_model
gc.collect()
torch.cuda.empty_cache()

MERGED_DIR = (
    ARTIFACTS_DIR
    / f"qlora_r{BEST_QLORA_RANK}"
    / "merged_fp16"
)

base_for_merge = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map={"": "cpu"},
    )
)

peft_model = PeftModel.from_pretrained(
    base_for_merge,
    QLORA_DIR,
)

merged_model = (
    peft_model.merge_and_unload()
)

merged_model.save_pretrained(
    MERGED_DIR,
    safe_serialization=True,
    max_shard_size="4GB",
)

tokenizer.save_pretrained(
    MERGED_DIR
)

print("Merge guardado en:", MERGED_DIR)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merge guardado en: /content/drive/MyDrive/EvidenceLab/EvidenceLab/artifacts/03_llama32_lora_qlora/qlora_r64/merged_fp16


# 9. Tamaños y métricas estructurales

In [24]:
def dir_size_gb(path):
    total = 0

    for item in Path(path).rglob("*"):
        if item.is_file():
            total += item.stat().st_size

    return total / 1024**3

print(
    "LoRA adapter GB:",
    dir_size_gb(LORA_DIR),
)
print(
    "QLoRA adapter GB:",
    dir_size_gb(QLORA_DIR),
)
print(
    "Merged QLoRA FP16 GB:",
    dir_size_gb(MERGED_DIR),
)

LoRA adapter GB: 0.10670960415154696
QLoRA adapter GB: 6.378778171725571
Merged QLoRA FP16 GB: 6.000339447520673


In [25]:
def parse_json_safe(value):
    if isinstance(value, dict):
        return value

    try:
        return json.loads(value)
    except Exception:
        return None

def structural_eval(df, pred_col):
    rows = []

    for _, row in df.iterrows():
        pred = parse_json_safe(
            row[pred_col]
        )
        exp = parse_json_safe(
            row["expected"]
        )

        valid = pred is not None

        if pred is not None and exp is not None:
            pred_keys = set(pred.keys())
            exp_keys = set(exp.keys())

            schema = pred_keys == exp_keys

            coverage = (
                len(pred_keys & exp_keys)
                / len(exp_keys)
                if exp_keys
                else 1.0
            )

            exact = pred == exp
        else:
            schema = False
            coverage = 0.0
            exact = False

        rows.append({
            "example_id": row["example_id"],
            "task": row["task"],
            "valid_json": valid,
            "schema_exact": schema,
            "key_coverage": coverage,
            "exact_json": exact,
        })

    return pd.DataFrame(rows)

lora_struct = structural_eval(
    lora_df,
    "lora_after",
)

qlora_struct = structural_eval(
    qlora_df,
    "qlora_after",
)

print("LoRA")
display(lora_struct)

print("QLoRA")
display(qlora_struct)

LoRA


,example_id,task,valid_json,schema_exact,key_coverage,exact_json
0,CASE-MX-006-VERIFY-001,claim_verification,True,True,1.0,True
1,CASE-MX-006-THEORY-001,theory_assessment,True,True,1.0,False
2,CASE-MX-006-CONTRA-001,contradiction_analysis,True,True,1.0,False
3,CASE-MX-006-RESP-001,responsibility_reasoning,True,True,1.0,False
4,CASE-MX-006-RECONSTRUCTION,case_reconstruction,True,True,1.0,False


QLoRA


,example_id,task,valid_json,schema_exact,key_coverage,exact_json
0,CASE-MX-006-VERIFY-001,claim_verification,True,True,1.0,True
1,CASE-MX-006-THEORY-001,theory_assessment,True,True,1.0,False
2,CASE-MX-006-CONTRA-001,contradiction_analysis,True,True,1.0,False
3,CASE-MX-006-RESP-001,responsibility_reasoning,True,True,1.0,False
4,CASE-MX-006-RECONSTRUCTION,case_reconstruction,True,True,1.0,False


# 10. Tabla LoRA vs QLoRA

In [26]:
comparison = pd.DataFrame([
    {
        "technique": "LoRA",
        "rank": LORA_RANK,
        "eval_loss": lora_eval_loss,
        "time_seconds": lora_time,
        "peak_vram_gb": lora_peak,
        "valid_json_rate":
            lora_struct[
                "valid_json"
            ].mean(),
        "schema_exact_rate":
            lora_struct[
                "schema_exact"
            ].mean(),
        "exact_json_rate":
            lora_struct[
                "exact_json"
            ].mean(),
    },
    {
        "technique": "QLoRA",
        "rank": BEST_QLORA_RANK,
        "eval_loss": qlora_eval_loss,
        "time_seconds": qlora_time,
        "peak_vram_gb": qlora_peak,
        "valid_json_rate":
            qlora_struct[
                "valid_json"
            ].mean(),
        "schema_exact_rate":
            qlora_struct[
                "schema_exact"
            ].mean(),
        "exact_json_rate":
            qlora_struct[
                "exact_json"
            ].mean(),
    },
])

display(comparison)

comparison.to_csv(
    ARTIFACTS_DIR
    / "lora_vs_qlora_metrics.csv",
    index=False,
)

,technique,rank,eval_loss,time_seconds,peak_vram_gb,valid_json_rate,schema_exact_rate,exact_json_rate
0,LoRA,16,0.156779,667.958232,8.425574,1.0,1.0,0.2
1,QLoRA,64,0.097523,784.347307,7.302553,1.0,1.0,0.2


In [1]:
import json, pandas as pd
from pathlib import Path

rescate = {}

# --- 1. TAMAÑOS EN DISCO: la rúbrica los pide y la celda 34 solo los imprime ---
def dir_size_gb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1024**3

rescate["sizes_gb"] = {
    "lora_adapter": dir_size_gb(LORA_DIR),
    "qlora_adapter": dir_size_gb(QLORA_DIR),
    "qlora_merged_fp16": dir_size_gb(MERGED_DIR),
}
print(json.dumps(rescate["sizes_gb"], indent=2))

# --- 2. EVALUACIÓN ESTRUCTURAL por ejemplo: la celda 35 solo la despliega ---
lora_struct.to_csv(ARTIFACTS_DIR / "lora_structural_eval.csv", index=False)
qlora_struct.to_csv(ARTIFACTS_DIR / "qlora_structural_eval.csv", index=False)

# --- 3. CONFIGURACIÓN, para poder reproducir y para el reporte ---
rescate["config"] = {
    "model": MODEL_NAME, "max_length": MAX_LENGTH,
    "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
    "best_qlora_rank": int(BEST_QLORA_RANK), "qlora_ranks": QLORA_RANKS,
    "target_modules": TARGET_MODULES,
    "lora_lr": LORA_LR, "qlora_lr": QLORA_LR,
    "epochs": FINAL_EPOCHS, "rank_sweep_max_steps": RANK_SWEEP_MAX_STEPS,
    "batch": TRAIN_BATCH, "grad_accum": GRAD_ACCUM,
    "train/val/test": [len(train_ds), len(val_ds), len(test_ds)],
    "gpu": GPU_NAME, "seed": SEED,
}

# --- 4. MÉTRICAS FINALES consolidadas ---
rescate["lora"]  = {"eval_loss": lora_eval_loss,  "time_s": lora_time,  "peak_vram_gb": lora_peak}
rescate["qlora"] = {"eval_loss": qlora_eval_loss, "time_s": qlora_time, "peak_vram_gb": qlora_peak}
rescate["rank_sweep"] = rank_df.to_dict("records")

with (ARTIFACTS_DIR / "run_summary.json").open("w", encoding="utf-8") as f:
    json.dump(rescate, f, ensure_ascii=False, indent=2, default=str)

print("\nGuardado run_summary.json")
print(pd.read_csv(ARTIFACTS_DIR / "lora_vs_qlora_metrics.csv"))

NameError: name 'LORA_DIR' is not defined

## Resultado final del notebook

Al terminar tendremos:
- LoRA normal;
- QLoRA NF4;
- sweep de ranks `4/8/16/32/64`;
- tiempos y VRAM;
- eval loss;
- adapters;
- modelo QLoRA mergeado;
- tamaños en disco;
- mismos cinco prompts;
- métricas de JSON/schema;
- artefactos listos para comparar con Phi Full SFT.